# Neural-LAM: Hello World Example

Welcome to the Neural-LAM "Hello World" example! This notebook provides a step-by-step guide for users to run a full model training and evaluation using a small subset of DANRA data.

This will walk you through installing the package, preparing the data, generating the graph, training the model, and evaluating the results. It is designed to showcase the capabilities of Neural-LAM for new contributors.

## 1. Environment Setup

We will install Neural-LAM and its dependencies using [PDM](https://pdm.fming.dev/), a modern Python package manager, along with `ipykernel` so we can run this notebook.

In [ ]:
# Install pdm if you haven't already
!pip install pdm

# Install Neural-LAM dependencies using pdm
!pdm install

# Add ipykernel for running this notebook
!pdm add -d ipykernel

## 2. Data Preparation

We will use a small subset of DANRA data for quick execution. Neural-LAM uses `mllam-data-prep` to fetch and preprocess data. The datastore configuration we use here (`tests/datastore_examples/mdp/danra_100m_winds/danra.datastore.yaml`) defines how the data is loaded and structured.

**Key Parameter:**
- `--config`: Points to the datastore YAML configuration file that defines datasets to read, variables to select, and how to split the data (train/test/val).

In [ ]:
import os
datastore_config = "../tests/datastore_examples/mdp/danra_100m_winds/danra.datastore.yaml"

# Preprocess the dataset to zarr format
!pdm run python -m mllam_data_prep --config {datastore_config}

## 3. Graph Generation

Next, we generate the graph structure required by the graph neural network. We will create a hierarchical graph suitable for the Hi-LAM model.

**Key Parameters:**
- `--config_path`: Points to the main Neural-LAM configuration file (`config.yaml`) which links to the datastore and defines the problem scope.
- `--name`: The name assigned to the generated graph, determining the folder name in the `graphs` directory.
- `--hierarchical`: Flag to generate a hierarchical graph instead of a flat multi-scale graph. This is required for `hi_lam` models.

In [ ]:
config_path = "../tests/datastore_examples/mdp/danra_100m_winds/config.yaml"

!pdm run python -m neural_lam.create_graph --config_path {config_path} --name helloworld_graph --hierarchical

## 4. Model Training

We can now train the model! We'll run a short training process on the CPU to quickly demonstrate the flow. We force CPU-only execution by setting `CUDA_VISIBLE_DEVICES=""` before our command so we don't accidentally consume a full GPU for a 1-epoch test.

**Key Parameters:**
- `--model`: Specifies the model architecture. We use `hi_lam` to match our hierarchical graph.
- `--graph`: Specifies the name of the graph we generated in the previous step (`helloworld_graph`).
- `--epochs`: Sets the upper limit on epochs. We use `1` here for a quick test.
- `--logger wandb`: Logs training progress to Weights & Biases (by default it uses wandb, you can switch to mlflow if configured).

In [ ]:
!CUDA_VISIBLE_DEVICES="" pdm run python -m neural_lam.train_model --config_path {config_path} --model hi_lam --graph helloworld_graph --epochs 1

## 5. Evaluation and Visualization (WandB)

Neural-LAM is fully integrated with Weights & Biases (W&B). During training, it records validation metrics, and when we evaluate on the test split, it generates and logs spatial error maps and sample prediction charts directly to the W&B dashboard using `neural_lam.vis`.

To generate these plots and metrics, use the `--eval test` flag as shown below.

*(Make sure you have logged into wandb using `!pdm run wandb login` if you want to see the online dashboard, otherwise results are saved to `./wandb/` locally)*

In [ ]:
# NOTE: You must provide the path to your newly generated checkpoint file.
# Check the 'saved_models/' directory for the exact path depending on your run name.
checkpoint_path = "saved_models/<your_run_name>/last.ckpt"

# Evaluate model on test data to generate metrics, maps, and charts:
# !CUDA_VISIBLE_DEVICES="" pdm run python -m neural_lam.train_model --config_path {config_path} --model hi_lam --graph helloworld_graph --eval test --load {checkpoint_path}

## 6. Additional Considerations for Scaling

When you are ready to train on a larger dataset (like the full DANRA or MEPS), consider the following tips for scaling to larger runs:

1. **Pre-process Data Offline:** Large datasets take time to prepare. Use `mllam-data-prep` with Dask distribution (e.g., `--dask-distributed-local-core-fraction 0.5`) on a powerful machine to generate the `.zarr` files fully beforehand.
2. **Use High-Performance Computing (HPC):** Remove the `CUDA_VISIBLE_DEVICES=""` mask to utilize your system's GPUs. Neural-LAM supports multi-GPU distributed training via PyTorch Lightning. If running on a SLURM cluster, ensure you set `--num_nodes` properly and allocate enough GPUs.
3. **Adjust Epochs and Patience:** You will likely need far more than `1` epoch. Use early stopping concepts by tracking the `val_mean_loss` on WandB and letting training run for several days if necessary.
4. **Experiment with Architectures:** We used `hi_lam` here, but you can also try the flat `graph_lam` or scaling up the GNN layers (`--processor_layers`) and hidden dimension dimensions (`--hidden_dim`).